In [ ]:
# System & IO
import os
import glob
import joblib
import random

# Data Manipulation
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# Statistics & Machine Learning Baselines
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import mean_squared_error, r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor

# PyTorch & Geometric
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
from torch_geometric.loader import DataLoader
from torch.utils.data import Dataset

# Visualization & Progress
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm, trange
import seaborn as sns
sns.set(style='whitegrid')

# Hardware Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# --- GLOBAL CONSTANTS ---
# The primary root for all data
BASE_DIR = '../Data/Models/Model_1' 

# Directory paths
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
PROCESSED_TRAIN_DIR = os.path.join(TRAIN_DIR, 'Processed')
PROCESSED_VAL_DIR = os.path.join(BASE_DIR, 'val', 'Processed')
PROCESSED_TEST_DIR = os.path.join(BASE_DIR, 'internal_test', 'Processed')

# Static Node Files
DF_1D_STATIC_PATH = os.path.join(TRAIN_DIR, '1d_nodes_static.csv')
DF_2D_STATIC_PATH = os.path.join(TRAIN_DIR, '2d_nodes_static.csv')

# Edge & Connection Files
PATH_1D_EDGES = os.path.join(TRAIN_DIR, '1d_edge_index.csv')
PATH_2D_EDGES = os.path.join(TRAIN_DIR, '2d_edge_index.csv')
PATH_CONNECTIONS = os.path.join(TRAIN_DIR, '1d2d_connections.csv')

# Search Patterns (Optional, for globbing)
TRAIN_PATTERN = os.path.join(PROCESSED_TRAIN_DIR, '*.pt')
VAL_PATTERN   = os.path.join(PROCESSED_VAL_DIR, '*.pt')
TEST_PATTERN  = os.path.join(PROCESSED_TEST_DIR, '*.pt')

# Join the directory and the search pattern in one go
TRAIN_PATTERN = os.path.join(BASE_DIR, 'train', 'Processed', '*.pt')
VAL_PATTERN   = os.path.join(BASE_DIR, 'val', 'Processed', '*.pt')
TEST_PATTERN  = os.path.join(BASE_DIR, 'internal_test', 'Processed', '*.pt')

# Model Save Paths
model_1_save_path = 'best_flood_model_1.pth'
model_2_save_path = 'best_flood_model_2.pth'

### Data Proccessing/Cleaning

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

def check_multicollinearity(df):
    """
    Cleans data, calculates VIF scores, and returns both VIF results 
    and the raw correlation matrix for external visualization.
    """
    # 1. Clean data: Drop non-numeric and ID columns
    X = df.drop(columns=['node_idx'], errors='ignore').select_dtypes(include=[np.number])
    X = X.replace([np.inf, -np.inf], np.nan).dropna()
    
    if X.empty:
        raise ValueError("No data left for analysis after cleaning!")

    # 2. Calculate Correlation Matrix
    corr_matrix = X.corr()

    # 3. Calculate VIF Scores
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
    vif_data = vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True)
    
    # Return both as a tuple
    return vif_data, corr_matrix
def get_edge_index(path, src_mapping, dst_mapping, src_col='from_node', dst_col='to_node'):
    df = pd.read_csv(path)
    
    # Debug print to help you if it fails again
    if src_col not in df.columns or dst_col not in df.columns:
        print(f"Error in {path}! Available columns: {df.columns.tolist()}")
        raise KeyError(f"Could not find {src_col} or {dst_col}")

    src = df[src_col].map(src_mapping).values
    dst = df[dst_col].map(dst_mapping).values
    
    # Check for mapping failures
    if pd.isna(src).any() or pd.isna(dst).any():
        print(f"Warning: Some IDs in {path} weren't in your mapping. Check your static node files!")
    
    edge_index_np = np.stack([src, dst])
    return torch.from_numpy(edge_index_np).to(torch.long)

In [ ]:
# 1. Load static data to establish the "Universe" of nodes
df_1d_static = pd.read_csv(DF_1D_STATIC_PATH)
df_2d_static = pd.read_csv(DF_2D_STATIC_PATH)

# 2. Create Mappings: {Original_ID: Integer_Index}
mapping_1d = {id: i for i, id in enumerate(df_1d_static['node_idx'])}
mapping_2d = {id: i for i, id in enumerate(df_2d_static['node_idx'])}

In [ ]:
print("--- 1D Static VIF Analysis ---")
# Capture both the DataFrame and the Matrix
vif_1d_df, matrix_1d = check_multicollinearity(df_1d_static)
print(vif_1d_df[vif_1d_df['VIF'] > 10]) 

print("\n--- 2D Static VIF Analysis ---")
vif_2d_df, matrix_2d = check_multicollinearity(df_2d_static)
print(vif_2d_df[vif_2d_df['VIF'] > 10])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_dual_correlation(matrix1, matrix2, name1="1D Features", name2="2D Features"):
    # Set global font sizes
    plt.rcParams.update({'font.size': 14})
    
    # 1. Setup Figure - Wider aspect ratio to accommodate the side scale
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 8), facecolor='#ffffff')
    
    # Heatmap Logic - Bigger annotations, no individual cbars
    heatmap_args = {
        'annot': True, 
        'fmt': ".2f", 
        'cmap': 'coolwarm', 
        'center': 0, 
        'linewidths': 0.8, 
        'cbar': False, 
        'annot_kws': {"size": 13, "fontweight": 'black'} 
    }

    # 2. Plot First Matrix (1D)
    mask1 = np.triu(np.ones_like(matrix1, dtype=bool))
    sns.heatmap(matrix1, mask=mask1, ax=ax1, **heatmap_args)
    ax1.set_title(name1.upper(), fontsize=22, fontweight='black', pad=15)
    
    # 3. Plot Second Matrix (2D)
    mask2 = np.triu(np.ones_like(matrix2, dtype=bool))
    sns.heatmap(matrix2, mask=mask2, ax=ax2, **heatmap_args)
    ax2.set_title(name2.upper(), fontsize=22, fontweight='black', pad=15)

    # 4. Final Polish - Large, Bold Axis Labels
    for ax in [ax1, ax2]:
        ax.tick_params(axis='both', which='major', labelsize=15)
        plt.setp(ax.get_xticklabels(), fontweight='bold', rotation=35, ha="right")
        plt.setp(ax.get_yticklabels(), fontweight='bold')

    # 5. Vertical Scale on the Right
    # Using 'fraction' and 'pad' to keep it tucked tightly to the right
    mappable = ax2.get_children()[0]
    cbar = fig.colorbar(mappable, ax=[ax1, ax2], orientation='vertical', fraction=0.02, pad=0.03)
    cbar.ax.tick_params(labelsize=14)
    cbar.outline.set_linewidth(1.5)

    plt.tight_layout()
    # Manual adjustment to ensure title and scale don't collide
    plt.subplots_adjust(right=0.92, wspace=0.2)
    plt.show()

# Run it
plot_dual_correlation(matrix_1d, matrix_2d)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1D Invert Elevation
axes[0].hist(df_1d_static['invert_elevation'], bins=50, alpha=0.7, color='blue')
axes[0].set_xlabel('Elevation (m)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('1D Invert Elevation Distribution')

# 2D Elevation
axes[1].hist(df_2d_static['elevation'], bins=50, alpha=0.7, color='green')
axes[1].set_xlabel('Elevation (m)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('2D Elevation Distribution')

plt.tight_layout()
plt.savefig('static_elevations.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1D Pruning
df_1d_static = df_1d_static.drop(columns=['surface_elevation', 'position_x', 'position_y'])

# 2D Pruning
df_2d_static = df_2d_static.drop(columns=['min_elevation', 'position_x', 'position_y'])

In [ ]:
# Check both—don't be lazy!
print("--- 1D Static VIF Analysis ---")
vif_1d = check_multicollinearity(df_1d_static)
print(vif_1d[vif_1d['VIF'] > 10]) # Only show the problematic ones

print("\n--- 2D Static VIF Analysis ---")
vif_2d = check_multicollinearity(df_2d_static)
print(vif_2d[vif_2d['VIF'] > 10])

print("\n--- Final Features ---")
print(f"1D Features: {df_1d_static.columns.tolist()}")
print(f"2D Features: {df_2d_static.columns.tolist()}")

# Ensure no NaNs are left to break the Scaler
df_1d_static = df_1d_static.fillna(0)
df_2d_static = df_2d_static.fillna(0)

In [ ]:
# Box Plot for Outliers (after filling NaNs)
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_1d_static[['invert_elevation', 'depth']].fillna(0))
plt.title('1D Static Features - Outlier Detection')
plt.savefig('outlier_detection.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# 1. Setup Aesthetics
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
bg_color = '#f8f9fa'
text_color = '#1a252f'
accent_color = '#d35400'  # Burnt Orange from your previous diagrams

fig, ax = plt.subplots(figsize=(10, 7), facecolor=bg_color)
ax.set_facecolor(bg_color)

# 2. Data for Plotting (Example: Invert vs Surface Elevation to show redundancy)
x = df_1d_static['invert_elevation']
y = df_1d_static['surface_elevation'] # Using the dropped column as the example

# 3. Calculate Correlation and Regression
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
line = slope * x + intercept

# 4. Plotting
# Scatter with edgecolors for depth
ax.scatter(x, y, alpha=0.6, edgecolors='w', linewidth=0.5, color='#1f618d', label='Data Points')

# Trendline
ax.plot(x, line, color=accent_color, lw=2.5, linestyle='--', label=f'Trendline ($R^2={r_value**2:.3f}$)')

# 5. Styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, linestyle='--', alpha=0.3)

# Bold Labels
ax.set_xlabel('Invert Elevation (m)', fontsize=12, fontweight='bold', color=text_color)
ax.set_ylabel('Surface Elevation (m)', fontsize=12, fontweight='bold', color=text_color)

# Header & Rationale Annotation
ax.text(0.5, 1.12, 'MULTICOLLINEARITY ANALYSIS: ELEVATION FEATURES', 
        transform=ax.transAxes, fontsize=18, fontweight='black', ha='center', color=text_color)

# This note explains the "Why"
rationale_text = (
    f"OBSERVATION: Correlation Coefficient $R$ = {r_value:.4f}\n"
    "RATIONALE: Extreme linear dependency (VIF > 10) detected.\n"
    "ACTION: 'surface_elevation' dropped to prevent feature redundancy."
)
ax.text(0.05, 0.85, rationale_text, transform=ax.transAxes, fontsize=10, 
        fontweight='bold', color=text_color, verticalalignment='top',
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='#bdc3c7', boxstyle='round,pad=0.5'))

plt.legend(frameon=True, facecolor='white', loc='lower right')
plt.savefig('multicollinearity_rationale.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Change line 17 to a column that STILL EXISTS
x = df_1d_static['invert_elevation']
y = df_1d_static['depth']  # This column is still in your DF

In [ ]:
# Replace the data setup lines (15-17) with this:
if 'surface_elevation' in df_1d_static.columns:
    x = df_1d_static['invert_elevation']
    y = df_1d_static['surface_elevation']
    title_suffix = "Correlation"
else:
    # If already dropped, we use a placeholder or depth to show the logic
    x = df_1d_static['invert_elevation']
    y = df_1d_static['depth']
    print("Note: 'surface_elevation' was already dropped due to high VIF.")
    title_suffix = "Proxy Correlation (Invert vs Depth)"

In [ ]:
# Load sample dynamic data for visualizations
sample_dyn = pd.read_csv(os.path.join(TRAIN_DIR, 'event_1', '1d_nodes_dynamic_all.csv'))

# Histogram for Dynamic Data (sample)
plt.figure(figsize=(10, 6))
sample_dyn['water_level'].hist(bins=50, alpha=0.7)
plt.xlabel('Water Level (m)')
plt.ylabel('Frequency')
plt.title('Sample Dynamic Water Level Distribution')
plt.show()

# Time-Series Plot (for one node)
node_data = sample_dyn[sample_dyn['node_idx'] == 0]
plt.plot(node_data['timestep'], node_data['water_level'])
plt.xlabel('Timestep')
plt.ylabel('Water Level (m)')
plt.title('Water Level Over Time (Node 0)')
plt.show()

In [ ]:
# Define the skeleton before the loop
edge_index_1d = get_edge_index(PATH_1D_EDGES, mapping_1d, mapping_1d)
edge_index_2d = get_edge_index(PATH_2D_EDGES, mapping_2d, mapping_2d)

edge_index_1d2d = get_edge_index(
    PATH_CONNECTIONS, mapping_1d, mapping_2d,
    src_col='node_1d', dst_col='node_2d' 
)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import Delaunay

# --- 1. SETUP DATA ---
# Synthetic 2D Positions (Shared coordinate space)
np.random.seed(42)
points = np.random.rand(60, 2) * 10  # 60 nodes in a 10x10 area

# Split into 1D and 2D nodes
pos_1d = {i: points[i] for i in range(20)}
pos_2d = {i: points[i] for i in range(20, 60)}

# Create 1D Pipe Edges (Linear chain)
edges_1d = [(i, i+1) for i in range(19)]
G_1d = nx.Graph()
G_1d.add_edges_from(edges_1d)

# Create 2D Surface Edges (Delaunay Mesh)
points_2d = points[20:]
tri = Delaunay(points_2d)
edges_2d = []
for simplex in tri.simplices:
    for i in range(3):
        # Offset indices back to original points range (20-60)
        u, v = simplex[i]+20, simplex[(i+1)%3]+20
        edges_2d.append((u, v))
G_2d = nx.Graph()
G_2d.add_edges_from(edges_2d)

# --- 2. PLOTTING ---
plt.figure(figsize=(14, 10), facecolor='#f8f9fa')
ax = plt.gca()

# 1. Draw 2D Surface Layer (The "Background" Mesh)
nx.draw_networkx_edges(
    G_2d, pos_2d, 
    width=1.5, 
    edge_color='#9b59b6', # Process Purple
    alpha=0.3, 
    ax=ax
)
nx.draw_networkx_nodes(
    G_2d, pos_2d, 
    node_size=60, 
    node_color='#8e44ad', 
    edgecolors='#ffffff', 
    linewidths=1.5, 
    ax=ax
)

# 2. Draw 1D Pipe Layer (The "Foreground" Network)
# Drawing this second naturally places it "on top"
nx.draw_networkx_edges(
    G_1d, pos_1d, 
    width=4.0,           # Extra thick for 1D prominence
    edge_color='#2c3e50', # Obsidian Navy
    alpha=0.8, 
    ax=ax
)
nx.draw_networkx_nodes(
    G_1d, pos_1d, 
    node_size=120, 
    node_color='#1f618d', # Deep Sea Blue
    edgecolors='#ffffff', 
    linewidths=2.5, 
    ax=ax
)

# --- 3. BOLD LABELS ---
ax.text(0.5, 0.96, 'Coupled 1D/2D Hydraulic Network Topology', 
        transform=ax.transAxes, fontsize=22, fontweight='black', 
        ha='center', color='#1a252f')

# Legend-style indicators (Manual text placement)
ax.text(0.05, 0.05, "● 1D Pipe Junctions", color='#1f618d', fontweight='black', transform=ax.transAxes)
ax.text(0.05, 0.02, "● 2D Surface Cells", color='#8e44ad', fontweight='black', transform=ax.transAxes)

ax.axis('off')
plt.savefig('fused_network_topology.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def process_flood_directory(source_path, folders, output_dir, edge_indices, scalers, dataset_tag="Train"):
    os.makedirs(output_dir, exist_ok=True)
    s1s, s2s, s1d, s2d = scalers
    e1, e2, e12 = edge_indices
    
    # Stats for assignment reporting - expanded for better visibility
    stats = {
        "total": len(folders), 
        "skipped_missing_val": 0, 
        "skipped_physical_violation": 0, 
        "skipped_shape_mismatch": 0,
        "skipped_empty_graph": 0,
        "success": 0
    }
    
    # Pre-scale static data to avoid redundant computation in the loop
    static_1d_scaled = s1s.transform(df_1d_static.drop(columns=['node_idx']))
    static_2d_scaled = s2s.transform(df_2d_static.drop(columns=['node_idx']))

    for folder in tqdm(folders, desc=f"Processing {dataset_tag}"):
        event_path = os.path.join(source_path, folder)
        try:
            # Load and sort to ensure temporal/spatial alignment
            df_1d_dyn = pd.read_csv(os.path.join(event_path, '1d_nodes_dynamic_all.csv')).sort_values(['timestep', 'node_idx'])
            df_2d_dyn = pd.read_csv(os.path.join(event_path, '2d_nodes_dynamic_all.csv')).sort_values(['timestep', 'node_idx'])
            df_time = pd.read_csv(os.path.join(event_path, 'timesteps.csv'))
            
            num_t = len(df_time)
            num_1d = static_1d_scaled.shape[0]
            num_2d = static_2d_scaled.shape[0]

            # --- 1. DATA INTEGRITY & SHAPE CHECK ---
            # Verify that we have the exact number of rows expected (T * N)
            expected_1d_rows = num_t * num_1d
            expected_2d_rows = num_t * num_2d
            
            if len(df_1d_dyn) != expected_1d_rows or len(df_2d_dyn) != expected_2d_rows:
                stats["skipped_shape_mismatch"] += 1
                continue

            # --- 2. QUALITY GATE (Missing Values) ---
            missing_ratio = max(df_1d_dyn.isnull().mean().max(), df_2d_dyn.isnull().mean().max())
            if missing_ratio > 0.10:
                stats["skipped_missing_val"] += 1
                continue

            # --- 3. PHYSICAL CONSISTENCY (e.g., Water depth cannot be negative) ---
            numeric_1d = df_1d_dyn.select_dtypes(include=np.number)
            if (numeric_1d < -1e-3).any().any(): # Small epsilon for float noise
                stats["skipped_physical_violation"] += 1
                continue

            # Impute remaining small holes
            df_1d_dyn = df_1d_dyn.fillna(0)
            df_2d_dyn = df_2d_dyn.fillna(0)

            # --- 4. TENSORIZATION & NORMALIZATION ---
            x1d = s1d.transform(df_1d_dyn.drop(columns=['node_idx', 'timestep']))
            x2d = s2d.transform(df_2d_dyn.drop(columns=['node_idx', 'timestep']))
            
            # Reshape to (Timesteps, Nodes, Features)
            x1d_3d = x1d.reshape(num_t, num_1d, -1)
            x2d_3d = x2d.reshape(num_t, num_2d, -1)

            # Broadcast static features across all timesteps
            x1s_3d = np.repeat(static_1d_scaled[np.newaxis, :, :], num_t, axis=0)
            x2s_3d = np.repeat(static_2d_scaled[np.newaxis, :, :], num_t, axis=0)

            # --- 5. HETERO DATA ENCAPSULATION ---
            event_data = HeteroData()
            
            # Combine static and dynamic features along the feature dimension (axis=-1)
            event_data['node1d'].x = torch.from_numpy(np.concatenate([x1s_3d, x1d_3d], axis=-1)).float()
            event_data['node2d'].x = torch.from_numpy(np.concatenate([x2s_3d, x2d_3d], axis=-1)).float()
            
            # Store topology
            event_data['node1d', 'pipe', 'node1d'].edge_index = e1
            event_data['node2d', 'surface', 'node2d'].edge_index = e2
            event_data['node1d', 'exchange', 'node2d'].edge_index = e12
            
            # Metadata for easy downstream filtering
            event_data.num_timesteps = num_t
            event_data.event_id = folder

            # --- 6. SAMPLE LOGGING ---
            if stats["success"] == 0 and dataset_tag == "Train":
                print(f"\n--- {dataset_tag} Sample Verified: {folder} ---")
                print(f"Nodes: 1D={num_1d}, 2D={num_2d} | Timesteps={num_t}")
                print(f"Edges: Pipe={e1.shape[1]}, Surface={e2.shape[1]}, Exchange={e12.shape[1]}")
                print(f"Feature Vector Size: 1D={event_data['node1d'].x.shape[-1]}")

            torch.save(event_data, os.path.join(output_dir, f'event_{folder}.pt'))
            stats["success"] += 1
            
        except FileNotFoundError:
            stats["skipped_missing_val"] += 1
        except ValueError as ve:
            # Catches reshape errors specifically
            stats["skipped_shape_mismatch"] += 1
        except Exception as e:
            print(f"Unexpected error in {folder}: {e}")
            stats["skipped_empty_graph"] += 1

    print(f"\n--- {dataset_tag} Final Report ---")
    for k, v in stats.items():
        print(f"{k.replace('_', ' ').title()}: {v}")

In [ ]:
# --- 1. SETUP & DATA LOADING ---
# We are now operating exclusively within the train_path for all splits
df_1d_static = pd.read_csv(DF_1D_STATIC_PATH)
df_2d_static = pd.read_csv(DF_2D_STATIC_PATH)

# VIF DROPS: Removing redundant features to ensure model stability
df_1d_static = df_1d_static.drop(columns=['surface_elevation', 'position_x', 'position_y'])
df_2d_static = df_2d_static.drop(columns=['min_elevation', 'position_x', 'position_y'])

# Define node counts for tensor reshaping
num_1d_nodes = len(df_1d_static)
num_2d_nodes = len(df_2d_static)

# --- 2. SCALER INITIALIZATION & FITTING ---
scaler_1d_static = StandardScaler()
scaler_2d_static = StandardScaler()
scaler_dyn_1d = StandardScaler()
scaler_dyn_2d = StandardScaler()

print("Fitting static scalers...")
scaler_1d_static.fit(df_1d_static.drop(columns=['node_idx']))
scaler_2d_static.fit(df_2d_static.drop(columns=['node_idx']))

# SPLIT LOGIC: 80/10/10 split of the 68 healthy events
all_train_folders = [f for f in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, f)) and f != 'Processed']
random.shuffle(all_train_folders)

train_idx = int(0.8 * len(all_train_folders))
val_idx = int(0.9 * len(all_train_folders))

train_folders = all_train_folders[:train_idx]
val_folders = all_train_folders[train_idx:val_idx]
test_folders = all_train_folders[val_idx:]

print(f"Split completed: {len(train_folders)} Train, {len(val_folders)} Val, {len(test_folders)} Test")

print("Fitting dynamic scalers on training split...")
sample_1d, sample_2d = [], []
for folder in train_folders[:20]:
    s1 = pd.read_csv(os.path.join(TRAIN_DIR, folder, '1d_nodes_dynamic_all.csv')).drop(columns=['node_idx', 'timestep'])
    s2 = pd.read_csv(os.path.join(TRAIN_DIR, folder, '2d_nodes_dynamic_all.csv')).drop(columns=['node_idx', 'timestep'])
    sample_1d.append(s1)
    sample_2d.append(s2)

scaler_dyn_1d.fit(pd.concat(sample_1d).fillna(0))
scaler_dyn_2d.fit(pd.concat(sample_2d).fillna(0))

all_scalers = (scaler_1d_static, scaler_2d_static, scaler_dyn_1d, scaler_dyn_2d)

# --- 3. EDGE CONSTRUCTION ---
mapping_1d = {id: i for i, id in enumerate(df_1d_static['node_idx'])}
mapping_2d = {id: i for i, id in enumerate(df_2d_static['node_idx'])}

e1 = get_edge_index(os.path.join(TRAIN_DIR, '1d_edge_index.csv'), mapping_1d, mapping_1d)
e2 = get_edge_index(os.path.join(TRAIN_DIR, '2d_edge_index.csv'), mapping_2d, mapping_2d)
e12 = get_edge_index(os.path.join(TRAIN_DIR, '1d2d_connections.csv'), mapping_1d, mapping_2d, src_col='node_1d', dst_col='node_2d')
edges = (e1, e2, e12)

# --- 4. THE COMPLETE PROCESSING FUNCTION ---


# --- 5. EXECUTION ---
datasets = [
    (train_folders, PROCESSED_TRAIN_DIR, "Train"),
    (val_folders, PROCESSED_VAL_DIR, "Validation"),
    (test_folders, PROCESSED_TEST_DIR, "Internal_Test")
]

for folders, out_dir, tag in datasets:
    process_flood_directory(TRAIN_DIR, folders, out_dir, edges, all_scalers, tag)

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load the pre-processed HeteroData object from disk
        return torch.load(self.file_paths[idx])

In [ ]:

# 1. Collect all processed .pt file paths for each split
# Using glob to grab every file in the Processed directories we just created
train_files = glob.glob(TRAIN_PATTERN)
val_files = glob.glob(VAL_PATTERN)
test_files = glob.glob(TEST_PATTERN)

# 2. Initialize the Datasets
# Assuming your FloodDataset class is designed to take a list of file paths
train_ds = FloodDataset(train_files)
val_ds = FloodDataset(val_files)
test_ds = FloodDataset(test_files)

# 3. Create the Loaders
# shuffle=True for training to ensure the model doesn't learn the order of storms
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

# Quick sanity check for your report
print("Loaders initialized:")
print(f" - Training Batches: {len(train_loader)}")
print(f" - Validation Batches: {len(val_loader)}")
print(f" - Test Batches: {len(test_loader)}")

In [ ]:
# Calculate average statistics across events for data understanding
import torch

# Load a few sample events to compute stats
sample_events = train_files  # First 5 training events
stats = {
    'num_timesteps': [],
    'max_water_level_1d': [],
    'mean_water_level_1d': [],
    'flooded_nodes_1d': [],  # Nodes with water_level > 0.5 scaled (simple flood threshold)
    'max_inlet_flow': [],
    'num_1d_nodes': [],
    'num_2d_nodes': []
}

for event_file in sample_events:
    data = torch.load(event_file, weights_only=False)  # HeteroData needs this
    num_t = data['node1d'].x.shape[0]  # Timesteps from shape
    stats['num_timesteps'].append(num_t)
    
    # 1D nodes: features are static (3) + dynamic (2) = 5, last 2: water_level, inlet_flow
    x1d = data['node1d'].x  # (timesteps, nodes, 5)
    water_level_scaled = x1d[:, :, -2]  # Water level
    inlet_flow_scaled = x1d[:, :, -1]   # Inlet flow
    
    stats['max_water_level_1d'].append(water_level_scaled.max().item())
    stats['mean_water_level_1d'].append(water_level_scaled.mean().item())
    stats['max_inlet_flow'].append(inlet_flow_scaled.max().item())
    
    # Flooded nodes: count nodes where max water_level > 0.5 scaled
    flooded = (water_level_scaled.max(dim=0)[0] > 0.5).sum().item()
    stats['flooded_nodes_1d'].append(flooded)
    
    stats['num_1d_nodes'].append(x1d.shape[1])
    stats['num_2d_nodes'].append(data['node2d'].x.shape[1])

# Compute averages
avg_stats = {k: sum(v)/len(v) for k, v in stats.items()}

print("Average Statistics Across Sample Events:")
for k, v in avg_stats.items():
    print(f"{k.replace('_', ' ').title()}: {v:.2f}")

# Example: Typical event has ~100 timesteps, max water level scaled ~0.8, etc.